In [ ]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text 

In [ ]:
# URL of minicipalities in Brazil
url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?orderBy=nome"

# Get the minicipalities
try:
    response = requests.get(
        url=url
    )
    
    if response.status_code != 200:
        raise Exception
    else:
        data = response.json()

        #Pandas to flatten 
        df = pd.json_normalize(data)
        df.to_parquet(
            f"data/raw/municipalities/municipalities.parquet",
            index=False
        )

except Exception as e:
    print("Error:", e)

In [ ]:
# Load dotenv and getting credentials
load_dotenv()
database_url = os.getenv("DATABASE_CONNECTION")

# Creating engine
engine = create_engine(database_url)

In [ ]:
# Creating schema bronze if not exists
with engine.connect() as conn:
    conn.execute(text("create schema if not exists bronze"))
    conn.commit()


In [ ]:
# Reading parque file
df = pd.read_parquet("data/raw/municipalities/municipalities.parquet")

In [ ]:
# Salve into bronze
df.to_sql(
    name= "municipalities",
    con= engine,
    schema= "bronze",
    if_exists= "replace",
    index= False
)